### README

From coord, get features needed for ML prediction.

### Requirements

In [ ]:
# Python libraries
"""
biopython==1.85
pandas==2.2.3
"""

# Outer softwares
"""
SVM-BPfinder-3M: Corvelo A, Hallegger M, Smith CW, Eyras E. Genome-wide association between branch point properties and alternative splicing. PLoS Comput Biol. 2010 Nov 24;6(11):e1001016. doi: 10.1371/journal.pcbi.1001016. PMID: 21124863; PMCID: PMC2991248.
maxentscan: Yeo G, Burge CB. Maximum entropy modeling of short sequence motifs with applications to RNA splicing signals. J Comput Biol. 2004;11(2-3):377-94. doi: 10.1089/1066527041410418. PMID: 15285897.
"""

In [ ]:
# Import libraries
import pandas as pd
import subprocess
import io
import tempfile
import os


# Outer softwares
bpfinder = '/PATH/TO/SVM-BPfinder-3M/svm_bpfinder.py'
maxentscan = '/PATH/TO/maxEntScan/fordownload'

### Constants

In [ ]:
# Input: path to file with sequence of minigenes
SEQ_PATH = '/PATH/TO/Minigene_list.xlsx' 
INTRON_COL = 'intron'
DNEXON_COL = 'downstream exon'

# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'
OUTPUT_PREFIX = 'minigene'


### Functions

In [ ]:
# CALCULATE SPLICE SITE SCORES

def maxentscan_score_3ss(sequences):
    
    # Create a temp file for input sequences
    with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_input:
        for seq in sequences:
            temp_input.write(seq + '\n')
        temp_input_name = temp_input.name

    try:
        script_path = os.path.join(maxentscan, 'score3.pl')

        # Run score3.pl from maxentscan_dir to ensure matrix files found
        result = subprocess.run(
            ['perl', script_path, temp_input_name],
            cwd=maxentscan,
            capture_output=True,
            text=True,
            check=True
        )

        # Parse scores: each line is "<sequence>\t<score>"
        scores = [float(line.strip().split('\t')[1]) for line in result.stdout.strip().split('\n')]

    finally:
        # Clean up temp file
        os.remove(temp_input_name)

    return scores

In [ ]:
# CALCULATE BRANCH POINT SCORES

# writes a fasta file with intron sequece, each intron is an identifier
def write_fasta(seq_dict, fasta_filename, trim=(0,0)):
    # trim: number of bases do remove from 5' and 3' end
    start_trim, end_trim = trim

    with open(fasta_filename, "w") as fasta_file:
        for intron_id, seq in seq_dict.items():
            seq = str(seq)
            
            # ignore seq shorter that trim
            if len(seq) <= start_trim + end_trim:
                continue
            
            trimmed_seq = seq[start_trim:-end_trim] if end_trim > 0 else seq[start_trim:]
            fasta_file.write(f">{intron_id}\n{trimmed_seq}\n")

# get BPfinder output from fasta file
# output is a dataframe showing only the best prediction of BP per intron
def find_bp(fasta_filename):
    
    # Run SVM-BPfinder
    cmd = f"{bpfinder} -i {fasta_filename} -s Hsap -l 100 -d 10"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Check for errors
    if result.returncode != 0:
        print("Error running SVM-BPfinder:", result.stderr)
        return None

    # Convert output into a DataFrame
    svm_output = pd.read_csv(io.StringIO(result.stdout), sep="\t")
    
    # Ensure `svm_scr` is numeric
    svm_output["svm_scr"] = pd.to_numeric(svm_output["svm_scr"], errors="coerce")

    # Select the row with the highest `svm_scr` for each intron
    svm_output_best = svm_output.loc[svm_output.groupby("seq_id")["svm_scr"].idxmax()]
    
    return svm_output_best

### Analysis

##### 1. Read input files

In [ ]:
# sequence table
data_df = pd.read_csv(SEQ_PATH)

# select only NAGNAG
data_df = data_df[data_df['NNAGNAG type'] != 'no NAGNAG'].copy()
data_df

##### 2. Get features: 3' end sequence

In [ ]:
# dictionary with intron seq + 5nt dw exon
seq_dict = (
    data_df[INTRON_COL] + data_df[DNEXON_COL].str[:5]
).to_dict()



# add seq information to the dataframe
for i in range(-15,-8):  # intronic part of the 3' end
    col_name = f"3end_{i+8}_base"  
    data_df[col_name] = data_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

for i in range(-8,0):  # exonic part of the 3' end
    col_name = f"3end_{i+9}_base"  
    data_df[col_name] = data_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

data_df

##### 3. Get features: splice site scores

In [ ]:
# get list of proximal and distal 3'ss for stregth analysis (20nt intron + 3nt exon)
proximal_3ss_list = data_df.index.map(lambda idx: (seq_dict[idx][-28:-5]) if idx in seq_dict else None)
distal_3ss_list = data_df.index.map(lambda idx: (seq_dict[idx][-25:-2]) if idx in seq_dict else None)

#calculate maxentscan scores
data_df['proximal_3ss_score'] = maxentscan_score_3ss(proximal_3ss_list)
data_df['distal_3ss_score'] = maxentscan_score_3ss(distal_3ss_list)

data_df

##### 4. Get features: branchpoint

In [ ]:
# Write separate FASTA files for short and long introns
write_fasta(seq_dict, "short_introns.fa", trim=(0, 8))
write_fasta(seq_dict, "long_introns.fa", trim=(0, 5))

# Run BPfinder
shortbp_df = find_bp('short_introns.fa')
longbp_df = find_bp('long_introns.fa')

# Merge information from BP if using proximal or distal sites
longbp_df['bp_same'] = 1
longbp_df['ss_dist'] = longbp_df['ss_dist']-3
longbp_df = longbp_df[['seq_id', 'ss_dist', 'bp_seq', 'bp_same']]
mergebp_df = shortbp_df.merge(longbp_df, on=['seq_id', 'ss_dist', 'bp_seq'], how='left')
mergebp_df['bp_same'] = mergebp_df['bp_same'].fillna(0)

# Split the 'bp_seq' column into 9 separate columns
mergebp_df['bp_seq'] = mergebp_df['bp_seq'].str.upper()
bpsplit_df = mergebp_df["bp_seq"].apply(lambda x: pd.Series(list(x)))
# Rename the columns for clarity
bpsplit_df.columns = [f"bp_{i+1}_base" for i in range(9)]
# Merge back with the bp DataFrame
mergebp_df = pd.concat([mergebp_df, bpsplit_df], axis=1)

# Merge BP information to the data dataframe
data_df['seq_id'] = data_df.index.tolist()
data_df = data_df.merge(mergebp_df, how="left", on="seq_id")
data_df = data_df.drop(columns=['bp_seq', 'seq_id'])

data_df

##### 5. Adjust and save final dataframe

In [ ]:
# Sequence related features adjustment
base_cols = [col for col in data_df.columns if col.endswith('_base')]
# Capitalize bases
data_df[base_cols] = data_df[base_cols].apply(lambda col: col.str.upper())
# Change T to U
data_df[base_cols] = data_df[base_cols].replace('T', 'U')

data_df

In [ ]:
# Save the final dataframe to a new CSV file
data_df.to_csv(f"{OUTPUT_DIR}/{OUTPUT_PREFIX}_features.csv", index=False)